In [19]:
!pip install crewai chromadb sentence-transformers pytesseract pillow pandas scikit-learn tesseract-ocr

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'tesseract-ocr' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tesseract-ocr'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [60 lines of output]
      /usr/local/lib/python3.12/dist-packages/setuptools/dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              ********************************************************************************
   

In [2]:
import re
import json
import pandas as pd
import chromadb
from PIL import Image
import pytesseract

In [9]:
def extract_text(image_path):
    image = Image.open(image_path)
    text = pytesseract.image_to_string(image)
    return text

In [8]:
def extract_customer_data(text):

    customer = {}

    patterns = {
        "name": r"Name[: ]+([A-Za-z ]+)",
        "pan": r"([A-Z]{5}[0-9]{4}[A-Z])"
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, text)

        customer[key] = match.group(1) if match else None

    return customer

In [10]:
sanction_data = [
    "John Smith involved in money laundering",
    "Global Trading blacklisted",
    "Mohammed Ali sanction list"
]

In [11]:
client = chromadb.Client()

collection = client.create_collection("aml")

In [12]:
for i, item in enumerate(sanction_data):

    collection.add(
        ids=[str(i)],
        documents=[item]
    )

In [22]:
sanction_names = [
    "John Smith",
    "Mohammed Ali",
    "Global Trading"
]

def aml_screen(name):

    for sanctioned in sanction_names:

        if name.lower() == sanctioned.lower():
            return True

    return False

In [23]:
def calculate_risk(customer, aml_match):

    score = 0
    evidence = []

    if aml_match:
        score += 80
        evidence.append("Potential AML Match")

    if not customer.get("pan"):
        score += 20
        evidence.append("PAN Missing")

    return score, evidence

In [15]:
def make_decision(score):

    if score < 30:
        return "APPROVED"

    elif score < 70:
        return "REVIEW"

    return "REJECTED"

In [16]:
def process_document(file_path):

    text = extract_text(file_path)

    customer = extract_customer_data(text)

    aml_result = aml_screen(
        customer.get("name", "")
    )

    score, evidence = calculate_risk(
        customer,
        aml_result
    )

    decision = make_decision(score)

    return {
        "customer": customer,
        "score": score,
        "decision": decision,
        "evidence": evidence
    }

In [24]:
result = process_document("sample_low_risk.jpg")

print(json.dumps(result, indent=4))

{
    "customer": {
        "name": "Ramesh Kumar",
        "pan": "FGHIJ5878K"
    },
    "score": 0,
    "decision": "APPROVED",
    "evidence": []
}
